In [1]:
%pip install -q --upgrade pageindex

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from dotenv import load_dotenv
from pageindex import PageIndexClient
import pageindex.utils as utils

load_dotenv()

# Get your PageIndex API key from https://dash.pageindex.ai/api-keys
PAGEINDEX_API_KEY = os.environ["PAGEINDEX_API_KEY"]
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

In [ ]:
import openai

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

async def call_llm(prompt, model="gpt-4.1", temperature=0):
    client = openai.AsyncOpenAI(api_key=OPENAI_API_KEY)
    response = await client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature
    )
    return response.choices[0].message.content.strip()

In [4]:
import os, requests

# You can also use our GitHub repo to generate PageIndex tree
# https://github.com/VectifyAI/PageIndex

pdf_url = "https://arxiv.org/pdf/2501.12948.pdf"
pdf_path = os.path.join("../data", pdf_url.split('/')[-1])
os.makedirs(os.path.dirname(pdf_path), exist_ok=True)

response = requests.get(pdf_url)
with open(pdf_path, "wb") as f:
    f.write(response.content)
print(f"Downloaded {pdf_url}")

doc_id = pi_client.submit_document(pdf_path)["doc_id"]
print('Document Submitted:', doc_id)

Downloaded https://arxiv.org/pdf/2501.12948.pdf
Document Submitted: pi-cmmgm6yeo00o009qnbxdd2azu


In [6]:
# ...existing code...
import time

max_wait_seconds = 300
poll_interval = 5
start = time.time()

while time.time() - start < max_wait_seconds:
    if pi_client.is_retrieval_ready(doc_id):
        tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
        print("Simplified Tree Structure of the Document:")
        utils.print_tree(tree)
        break

    print("Still processing...")
    time.sleep(poll_interval)
else:
    print("Timed out waiting for PageIndex processing.")
# ...existing code...

Simplified Tree Structure of the Document:
[{'title': 'Abstract', 'node_id': '0000', 'summary': 'This text discusses the challenge of gen...'},
 {'title': '1 Introduction',
  'node_id': '0001',
  'summary': 'This text discusses the development of r...'},
 {'title': '2 DeepSeek-R1-Zero',
  'node_id': '0002',
  'summary': 'The text details the training of DeepSee...'},
 {'title': '3. DeepSeek-R1',
  'node_id': '0003',
  'summary': 'This text introduces DeepSeek-R1, an imp...'},
 {'title': '4 Experiment',
  'node_id': '0004',
  'summary': 'The text details the experimental evalua...'},
 {'title': '5 Ethics and Safety Statement',
  'node_id': '0005',
  'summary': 'The text addresses the ethical risks of ...'},
 {'title': '6 Conclusion, Limitation, and Future Wor...',
  'node_id': '0006',
  'summary': 'The text introduces DeepSeek-R1-Zero and...'},
 {'title': '7 Author List',
  'node_id': '0007',
  'summary': 'The text presents an author list organiz...'},
 {'title': 'Appendix A Background'

In [7]:
import json

query = "Explain training details of DeepSeek-R1-Zero?"

tree_without_text = utils.remove_fields(tree.copy(), fields=['text'])

search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

tree_search_result = await call_llm(search_prompt)

In [8]:
node_map = utils.create_node_mapping(tree)
tree_search_result_json = json.loads(tree_search_result)

print('Reasoning Process:')
utils.print_wrapped(tree_search_result_json['thinking'])

print('\nRetrieved Nodes:')
for node_id in tree_search_result_json["node_list"]:
    node = node_map[node_id]
    print(f"Node ID: {node['node_id']}\t Page: {node['page_index']}\t Title: {node['title']}")

Reasoning Process:
The question asks for the training details of DeepSeek-R1-Zero. The most relevant nodes will be
those that specifically describe the training process, algorithms, data, hyperparameters, and
infrastructure for DeepSeek-R1-Zero. From the tree, node '0002' (section 2 DeepSeek-R1-Zero)
provides a high-level summary of the training approach, including the use of RL and GRPO. Appendix B
(node '0009' and its children) is titled 'Training Details' and contains detailed subsections on RL
infrastructure (0010), reward model prompt (0011), data recipe (0012), hyperparameters (0014), and
reward hacking (0015), all of which are likely to contain granular training details. Appendix A
(0008) provides background on the RL algorithms (GRPO vs PPO) and the rationale for the training
approach, which is also relevant. Node '0017' (Appendix C) discusses the self-evolution of
DeepSeek-R1-Zero during training, which may include insights into the training process and its
effects. Node '0014

In [9]:
node_list = json.loads(tree_search_result)["node_list"]
relevant_content = "\n\n".join(node_map[node_id]["text"] for node_id in node_list)

print('Retrieved Context:\n')
utils.print_wrapped(relevant_content[:1000] + '...')

Retrieved Context:

# 2 DeepSeek-R1-Zero

We begin by elaborating on the training of DeepSeek-R1-Zero, which relies exclusively on
reinforcement learning without supervised fine-tuning. To facilitate large-scale RL efficiency, we
adopt Group Relative Policy Optimization (GRPO) *(Shao et al., 2024)*.

## 2.1 Group Relative Policy Optimization

GRPO *(Shao et al., 2024)* is the reinforcement learning algorithm that we adopt to train
DeepSeek-R1-Zero and DeepSeek-R1. It was originally proposed to simplify the training process and
reduce the resource consumption of Proximal Policy Optimization (PPO) *(Schulman et al., 2017)*,
which is widely used in the RL stage of LLMs *(Ouyang et al., 2022)*.

For each question $q$, GRPO samples a group of outputs $\{o_{1},o_{2},\cdots,o_{G}\}$ from the old
policy $\pi_{\theta_{old}}$ and then optimizes the policy model $\pi_{\theta}$ by maximizing the
following objective:

$\mathcal{J}_{GRPO}(\theta)=\mathbb{E}[q\sim P(Q),\{o_{i}\}_{i=1}^{G}\sim\pi_{\th

In [10]:
answer_prompt = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a clear, concise answer based only on the context provided.
"""

print('Generated Answer:\n')
answer = await call_llm(answer_prompt)
utils.print_wrapped(answer)

Generated Answer:

**Training Details of DeepSeek-R1-Zero**

DeepSeek-R1-Zero is a large language model trained **exclusively with reinforcement learning (RL)**,
without any supervised fine-tuning (SFT). Its training is designed to maximize reasoning ability,
especially for mathematics, coding, STEM, and logic tasks. Below are the key training details:

---

### 1. **RL Algorithm: Group Relative Policy Optimization (GRPO)**
- **GRPO** is used instead of the more common PPO. GRPO:
  - Samples a group of outputs per question from the old policy.
  - Computes the advantage for each output based on group rewards (normalized within the group).
  - Optimizes the policy by maximizing a clipped objective and penalizing KL divergence from a
reference policy.
  - **No value model** is used (unlike PPO), reducing memory and computation.
  - The reference policy is updated every 400 steps to the latest policy.

---

### 2. **Training Hyperparameters**
- **Learning rate:** 3e-6
- **KL coefficient:*